In [1]:
import numpy as np
import random

from qiskit import transpile
from qiskit.circuit import Parameter,ParameterExpression
from qiskit_algorithms import NumPyMinimumEigensolver
from qiskit.circuit.library import QAOAAnsatz
from qiskit_ibm_runtime import Session, EstimatorV2 as Estimator
from qiskit.converters import circuit_to_dag, dag_to_circuit
from qiskit_optimization.applications import Knapsack
from qiskit_optimization.converters import QuadraticProgramToQubo
from qiskit.circuit.library import QAOAAnsatz


import sys
sys.path.append("../")
from clapton.clapton import claptonize
from clapton.circuit_manipulation import transform_to_allowed_gates,qiskit_to_stim, modify_circuit, multi_angle_qaoa_circuit, generate_qiskit_param_map
from testing_scripts.energy_utils import evaluate_energy

import warnings
warnings.filterwarnings('ignore', category=DeprecationWarning)

In [2]:
prob = Knapsack(values=[3, 4, 5, 6, 7], weights=[2, 3, 4, 5, 6], max_weight=10)
qp = prob.to_quadratic_program()
print(qp.prettyprint())

# intermediate QUBO form of the optimization problem
conv = QuadraticProgramToQubo()
qubo = conv.convert(qp)

# qubit Hamiltonian and offset
op, offset = qubo.to_ising()
print(f"num qubits: {op.num_qubits}, offset: {offset}\n")
print(op)

cost_hamiltonian = op
paulis,coeffs = cost_hamiltonian.paulis.to_labels(),cost_hamiltonian.coeffs.real
reversed_paulis = [p[::-1] for p in paulis]

Problem name: Knapsack

Maximize
  3*x_0 + 4*x_1 + 5*x_2 + 6*x_3 + 7*x_4

Subject to
  Linear constraints (1)
    2*x_0 + 3*x_1 + 4*x_2 + 5*x_3 + 6*x_4 <= 10  'c0'

  Binary variables (5)
    x_0 x_1 x_2 x_3 x_4

num qubits: 9, offset: 1417.5

SparsePauliOp(['IIIIIIIIZ', 'IIIIIIIZI', 'IIIIIIZII', 'IIIIIZIII', 'IIIIZIIII', 'IIIZIIIII', 'IIZIIIIII', 'IZIIIIIII', 'ZIIIIIIII', 'IIIIIIIZZ', 'IIIIIIZIZ', 'IIIIIZIIZ', 'IIIIZIIIZ', 'IIIZIIIIZ', 'IIZIIIIIZ', 'IZIIIIIIZ', 'ZIIIIIIIZ', 'IIIIIIZZI', 'IIIIIZIZI', 'IIIIZIIZI', 'IIIZIIIZI', 'IIZIIIIZI', 'IZIIIIIZI', 'ZIIIIIIZI', 'IIIIIZZII', 'IIIIZIZII', 'IIIZIIZII', 'IIZIIIZII', 'IZIIIIZII', 'ZIIIIIZII', 'IIIIZZIII', 'IIIZIZIII', 'IIZIIZIII', 'IZIIIZIII', 'ZIIIIZIII', 'IIIZZIIII', 'IIZIZIIII', 'IZIIZIIII', 'ZIIIZIIII', 'IIZZIIIII', 'IZIZIIIII', 'ZIIZIIIII', 'IZZIIIIII', 'ZIZIIIIII', 'ZZIIIIIII'],
              coeffs=[-258.5+0.j, -388. +0.j, -517.5+0.j, -647. +0.j, -776.5+0.j, -130. +0.j,
 -260. +0.j, -520. +0.j, -390. +0.j,   78. +0.j,  104. +0.j, 

In [3]:
circuit = QAOAAnsatz(cost_operator=cost_hamiltonian, reps=1)

In [4]:
# Transform qiskit circ. to stim.
modified_circ = modify_circuit(circuit)
pcirc = transform_to_allowed_gates(modified_circ)

In [5]:
def parameter_renaming(dec_circ):
    dag = circuit_to_dag(dec_circ)
    gamma_counter, beta_counter = 0, 0
    angle_multipliers = {}
    for node in dag.op_nodes():
        if node.op.params and isinstance(node.op.params[0], ParameterExpression):
            param_name = list(node.op.params[0].parameters)[0].name 
            if "β" in param_name:

                multiplier = float(str(node.op.params[0]).split("*")[0])

                beta = Parameter(f'{multiplier}*beta_{beta_counter}')
                new_params = [beta if (isinstance(p, ParameterExpression)) else p for p in node.op.params]
                new_op = node.op.copy()
                new_op.params = new_params  # Create a modified version of the operation
                dag.substitute_node(node, new_op)  # Replace the node in the DAG
                beta_counter += 1

                angle_multipliers[beta.name] = multiplier
            

            elif "γ" in param_name:

                multiplier = float(str(node.op.params[0]).split("*")[0])
                
                gamma = Parameter(f'{multiplier}*gamma_{gamma_counter}')
                new_params = [gamma if (isinstance(p, ParameterExpression)) else p for p in node.op.params]
                new_op = node.op.copy()
                new_op.params = new_params  # Create a modified version of the operation
                dag.substitute_node(node, new_op)  # Replace the node in the DAG
                gamma_counter += 1

                angle_multipliers[gamma.name] = multiplier

    # Convert DAG back to a circuit
    new_qc = dag_to_circuit(dag)
    return new_qc, circuit_to_dag(new_qc) , angle_multipliers

# Usage
pcirc, dag , angle_multipliers= parameter_renaming(pcirc)

In [6]:
stim_circ = qiskit_to_stim(pcirc)
stim_circ.stim_circuit().diagram()

q0: -H-I---@---@---@---@---------@---@---------------@---@---------------------@---@---------------------------@---@---------------------------------@---@---------------------------------------@---@-S-SQRT_X-I-SQRT_X-S-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
           |   |   |   |         |   |               |   |                     |   |                           |   |                                 |   |                                       |   |
q1: ---H-I-X-I-X---|---|-@---@---|---|-@---@---------|---|-@---@---------------|---|-@---@---------------------|---|-@---@---------------------------|---|-@---@---------------------------------|---|-------------------@---@-S-SQRT_X-I-SQRT_X-S-------------------------------------------------------------------------------------------------------------------------------------------------------------------
                   |   | |   |   |   | |   |         |   | |   |               |   | |   |                     |   | |   |                           |   | |   |                                 |   |                   |   |
q2: -----------H-I-X-I-X-X-I-X---|---|-|---|-@---@---|---|-|---|-@---@---------|---|-|---|-@---@---------------|---|-|---|-@---@---------------------|---|-|---|-@---@---------------------------|---|-------------------|---|-------------------@---@-S-SQRT_X-I-SQRT_X-S-------------------------------------------------------------------------------------------------------------------------------------------
                                 |   | |   | |   |   |   | |   | |   |         |   | |   | |   |               |   | |   | |   |                     |   | |   | |   |                           |   |                   |   |                   |   |
q3: -------------------------H-I-X-I-X-X-I-X-X-I-X---|---|-|---|-|---|-@---@---|---|-|---|-|---|-@---@---------|---|-|---|-|---|-@---@---------------|---|-|---|-|---|-@---@---------------------|---|-------------------|---|-------------------|---|-------------------@---@-S-SQRT_X-I-SQRT_X-S-------------------------------------------------------------------------------------------------------------------
                                                     |   | |   | |   | |   |   |   | |   | |   | |   |         |   | |   | |   | |   |               |   | |   | |   | |   |                     |   |                   |   |                   |   |                   |   |
q4: ---------------------------------------------H-I-X-I-X-X-I-X-X-I-X-X-I-X---|---|-|---|-|---|-|---|-@---@---|---|-|---|-|---|-|---|-@---@---------|---|-|---|-|---|-|---|-@---@---------------|---|-------------------|---|-------------------|---|-------------------|---|-------------------@---@-S-SQRT_X-I-SQRT_X-S-------------------------------------------------------------------------------------------
                                                                               |   | |   | |   | |   | |   |   |   | |   | |   | |   | |   |         |   | |   | |   | |   | |   |               |   |                   |   |                   |   |                   |   |                   |   |
q5: -----------------------------------------------------------------------H-I-X-I-X-X-I-X-X-I-X-X-I-X-X-I-X---|---|-|---|-|---|-|---|-|---|-@---@---|---|-|---|-|---|-|---|-|---|-@---@---------|---|-------------------|---|-------------------|---|-------------------|---|-------------------|---|-------------------@---@-S-SQRT_X-I-SQRT_X-S-------------------------------------------------------------------
                                                                                                               |   | |   | |   | |   | |   | |   |   |   | |   | |   | |   | |   | |   |         |   |                   |   |                   |   |                   |   |                   |   |                   |   |
q6: ------

In [7]:
# Parameter Mapping
param_map = generate_qiskit_param_map(pcirc)

In [8]:
stim_circ.define_parameter_map(param_map)

In [ ]:
# we can perform CAFQA by using the main optimization function "claptonize"

ks_best, _, energy_best = claptonize(
    reversed_paulis,
    coeffs,
    stim_circ,
    n_proc=4,           # total number of processes in parallel
    n_starts=4,         # number of random genetic algorithm starts in parallel
    n_rounds=1,         # number of budget rounds, if None it will terminate itself
    callback=print,     # callback for internal parameter (#iteration, energies, ks) processing
    budget=10         # budget per genetic algorithm instance
)

STARTING ROUND 0


started GA at id 1 with 1 procs

started GA at id 2 with 1 procs


started GA at id 3 with 1 procs


/global/homes/d/dhanvib/.conda/envs/qaoa_w_sage/lib/python3.10/site-packages/pygad/pygad.py:1139: UserWarning: The 'delay_after_gen' parameter is deprecated starting from PyGAD 3.3.0. To delay or pause the evolution after each generation, assign a callback function/method to the 'on_generation' parameter to adds some time delay.
  warnings.warn("The 'delay_after_gen' parameter is deprecated starting from PyGAD 3.3.0. To delay or pause the evolution after each generation, assign a callback function/method to the 'on_generation' parameter to adds some time delay.")


GA parameters used for this experiment:
  num_generations=100
  num_parents_mating=20
  population_size=100
  num_genes=54
  parent_selection_type=tournament
  keep_parents=-1
  crossover_type=single_point
  mutation_type=adaptive
  crossover_probability=0.9
  mutation_probability=(0.25, 0.01)
  keep_elitism=5


/global/homes/d/dhanvib/.conda/envs/qaoa_w_sage/lib/python3.10/site-packages/pygad/pygad.py:1139: UserWarning: The 'delay_after_gen' parameter is deprecated starting from PyGAD 3.3.0. To delay or pause the evolution after each generation, assign a callback function/method to the 'on_generation' parameter to adds some time delay.
  warnings.warn("The 'delay_after_gen' parameter is deprecated starting from PyGAD 3.3.0. To delay or pause the evolution after each generation, assign a callback function/method to the 'on_generation' parameter to adds some time delay.")


GA parameters used for this experiment:
  num_generations=100
  num_parents_mating=20
  population_size=100
  num_genes=54
  parent_selection_type=tournament
  keep_parents=-1
  crossover_type=single_point
  mutation_type=adaptive
  crossover_probability=0.9
  mutation_probability=(0.25, 0.01)
  keep_elitism=5
started GA at id None with 1 procs

GA parameters used for this experiment:
  num_generations=100
  num_parents_mating=20
  population_size=100
  num_genes=54
  parent_selection_type=tournament
  keep_parents=-1
  crossover_type=single_point
  mutation_type=adaptive
  crossover_probability=0.9
  mutation_probability=(0.25, 0.01)
  keep_elitism=5


/global/homes/d/dhanvib/.conda/envs/qaoa_w_sage/lib/python3.10/site-packages/pygad/pygad.py:1139: UserWarning: The 'delay_after_gen' parameter is deprecated starting from PyGAD 3.3.0. To delay or pause the evolution after each generation, assign a callback function/method to the 'on_generation' parameter to adds some time delay.
  warnings.warn("The 'delay_after_gen' parameter is deprecated starting from PyGAD 3.3.0. To delay or pause the evolution after each generation, assign a callback function/method to the 'on_generation' parameter to adds some time delay.")
/global/homes/d/dhanvib/.conda/envs/qaoa_w_sage/lib/python3.10/site-packages/pygad/pygad.py:1139: UserWarning: The 'delay_after_gen' parameter is deprecated starting from PyGAD 3.3.0. To delay or pause the evolution after each generation, assign a callback function/method to the 'on_generation' parameter to adds some time delay.
  warnings.warn("The 'delay_after_gen' parameter is deprecated starting from PyGAD 3.3.0. To delay 

GA parameters used for this experiment:
  num_generations=100
  num_parents_mating=20
  population_size=100
  num_genes=54
  parent_selection_type=tournament
  keep_parents=-1
  crossover_type=single_point
  mutation_type=adaptive
  crossover_probability=0.9
  mutation_probability=(0.25, 0.01)
  keep_elitism=5
[0, array([-390., -195., -195.,    0.]), array([3, 3, 0, 2, 3, 3, 2, 3, 2, 1, 1, 2, 1, 0, 2, 1, 2, 0, 0, 2, 3, 0,
       2, 3, 2, 1, 3, 3, 2, 0, 0, 0, 3, 0, 3, 2, 1, 2, 0, 1, 1, 1, 1, 3,
       0, 0, 2, 3, 0, 2, 2, 0, 2, 1], dtype=object)]
[0, array([-390., -195., -195.,    0.]), array([3, 3, 0, 2, 3, 3, 2, 3, 2, 1, 1, 2, 1, 0, 2, 1, 2, 0, 0, 2, 3, 0,
       2, 3, 2, 1, 3, 3, 2, 0, 0, 0, 3, 0, 3, 2, 1, 2, 0, 1, 1, 1, 1, 3,
       0, 0, 2, 3, 0, 2, 2, 0, 2, 1], dtype=object)]
[0, array([-390., -195., -195.,    0.]), array([3, 3, 0, 2, 3, 3, 2, 3, 2, 1, 1, 2, 1, 0, 2, 1, 2, 0, 0, 2, 3, 0,
       2, 3, 2, 1, 3, 3, 2, 0, 0, 0, 3, 0, 3, 2, 1, 2, 0, 1, 1, 1, 1, 3,
       0, 0, 2, 3, 0,

In [10]:
energy_best

np.float64(-1139.0)

In [11]:
stim_circ.assign(ks_best)

In [12]:
# Solve with classical Eigensolver for comparison
eigensolver = NumPyMinimumEigensolver()
exact_solution = eigensolver.compute_minimum_eigenvalue(cost_hamiltonian).eigenvalue.real
print("Exact Energy from Eigensolver:", exact_solution)

Exact Energy from Eigensolver: -1430.5


In [13]:
cafqa_angles = [param * np.pi/2 for param in ks_best]

In [14]:
energies = [evaluate_energy(pcirc, cost_hamiltonian, cafqa_angles) for _ in range(10)]
average_energy = np.mean(energies)
print(f"Average CAFQA Qiskit Energy: {average_energy}")

Average CAFQA Qiskit Energy: -1138.2248291015626


In [15]:
# Random Initalization 
random_angles = np.random.random(len(ks_best))
random_energies = [evaluate_energy(pcirc, cost_hamiltonian, random_angles) for _ in range(10)]
min_energy = min(random_energies)
print(f"Minimum Energy found with Random initialization over 100 runs: {min_energy}")

Minimum Energy found with Random initialization over 100 runs: 1040.05859375
